# CMA-ES

In [3]:
# ! pip install cma "gymnasium[box2d]>=1.0" "stable-baselines3>=2.3"

In [4]:
import numpy as np
import cma
import matplotlib.pyplot as plt
import plotly.express as px
import pandas as pd
import gymnasium
import torch
import torch.nn as nn
from matplotlib.patches import Ellipse
from matplotlib.animation import FuncAnimation
from typing import Callable

### Optimization Problems

This cell defines three common benchmark functions, Sphere, Rosenbrock, and Rastrigin, used to test optimization algorithms. We also used these functions earlier to evaluate Adam and Momentum.

In [5]:
def sphere(x: np.ndarray) -> float:
    return float(np.sum(x**2))


def rosenbrock(x: np.ndarray) -> float:
    return float(np.sum(100.0 * (x[1:] - x[:-1] ** 2.0) ** 2.0 + (1.0 - x[:-1]) ** 2.0))


def rastrigin(x: np.ndarray) -> float:
    A: float = 10.0
    return float(A * len(x) + np.sum(x**2 - A * np.cos(2 * np.pi * x)))

### Helper functions

Convergence plot and animation.

In [6]:
def plot_convergence(histories: dict[str, list[float]], use_log_scale: bool = False) -> None:
    plt.figure()
    for name, fitness in histories.items():
        plt.plot(fitness, label=name)
    plt.xlabel("Generation")
    plt.ylabel("Best Fitness")
    plt.title("CMA-ES Convergence")
    plt.legend()
    plt.grid(True)
    if use_log_scale:
        plt.yscale("log")
    plt.show()

def animate_distribution(
    func: Callable[[np.ndarray], float],
    mu_history: list[np.ndarray],
    cov_history: list[np.ndarray],
    bounds: tuple[float, float] = (-5, 5),
    frames: int = 50,
    filename: str = "cma_es_animation.mp4",
) -> None:
    x = np.linspace(bounds[0], bounds[1], 200)
    y = np.linspace(bounds[0], bounds[1], 200)
    X, Y = np.meshgrid(x, y)
    coords = np.vstack([X.ravel(), Y.ravel()]).T
    Z = np.array([func(pt) for pt in coords]).reshape(X.shape)

    fig, ax = plt.subplots()
    levels = np.logspace(np.log10(Z.min() + 1e-6), np.log10(Z.max()), 30)

    def update(frame_index: int) -> None:
        ax.clear()
        ax.contour(X, Y, Z, levels=levels, cmap="viridis")
        mu = mu_history[frame_index]
        cov = cov_history[frame_index]
        vals, vecs = np.linalg.eigh(cov)
        angle = float(np.degrees(np.arctan2(vecs[0, 1], vecs[0, 0])))
        width, height = 2 * np.sqrt(vals)
        ellipse = Ellipse(
            xy=mu,
            width=width,
            height=height,
            angle=angle,
            edgecolor="red",
            facecolor="none",
            lw=2,
        )
        ax.add_patch(ellipse)
        ax.plot(mu[0], mu[1], "ro")
        ax.set_title(f"Generation {frame_index}")
        ax.set_xlim(bounds)
        ax.set_ylim(bounds)

    anim = FuncAnimation(fig, update, frames=min(frames, len(mu_history)), interval=200)
    anim.save(filename, writer="ffmpeg")
    print(f"Animation saved to {filename}")

### Running CMA-ES

In [16]:
from dataclasses import dataclass

@dataclass
class CMAESResult:
    best_fitness: list[float]
    mean_history: list[np.ndarray]
    cov_history: list[np.ndarray]
    sigma_history: list[float]

def run_cma_es(
    func: Callable[[np.ndarray], float],
    x0: np.ndarray,
    sigma0: float,
    max_iterations: int = 100
) -> CMAESResult:
    # TODO: Implement this function
    # TODO: Use cma.CMAEvolutionStrategy from pycma: https://pypi.org/project/cma/
    # Hint: Use ask and tell methods
    es = cma.CMAEvolutionStrategy(x0=x0,sigma0=sigma0)

    best_fitness = []
    mean_history = []
    cov_history = []
    sigma_history = []

    for _ in range(max_iterations):
        curr_solutions = es.ask()
        curr_fitted_values = [func(x) for x in curr_solutions]

        es.tell(curr_solutions, curr_fitted_values)

        mean_history.append(es.mean.copy())
        cov_history.append(es.C.copy())
        best_fitness.append(es.best.f)
        sigma_history.append(es.sigma)

    return CMAESResult(
        best_fitness,
        mean_history,
        cov_history,
        sigma_history
    )

### Ex. 1: Impact of the Starting Point
1.	Choose Rosenbrock in 2D.
2.	Run CMA-ES from at least five widely separated initial means (e.g., [-4,-4], [-1,3], [5,5]).
3.	Plot convergence curves and report:
- best fitness vs. generation,
- total evaluations to reach $f(x)\lt10^{-8}$ (or termination).
4.	Briefly discuss sensitivity to the start point. Prepare GIFs for two different starting points.

Hint: use `sigma0 = 0.5` and `max_iter = 250`.

In [17]:
result_dict={}
initial_means = [
    np.array([-4.0, -4.0]),
    np.array([-1.0, 3.0]),
    np.array([1.0, -3.0]),
    np.array([2.0, 4.0]),
    np.array([5.0, 5.0]),
]

for x0 in initial_means:
    result = run_cma_es(func=rosenbrock, x0=x0, sigma0=0.5, max_iterations=250)
    result_dict[str(x0)] = result.best_fitness

(3_w,6)-aCMA-ES (mu_w=2.0,w_1=63%) in dimension 2 (seed=212401, Thu May  7 10:22:22 2026)


CMAESResult(best_fitness=[15747.362640236213, 9868.145888903218, 2867.9723269506503, 2325.6037643073687, 2325.6037643073687, 1579.6123895535025, 1334.5856464926205, 929.0816985267962, 363.98995523139985, 125.27572541028066, 17.3114444989362, 15.764010559470865, 4.313775945405923, 4.313775945405923, 4.313775945405923, 4.313775945405923, 4.313775945405923, 4.313775945405923, 0.8959168647571463, 0.8277502035722772, 0.8277502035722772, 0.8277502035722772, 0.6248498146341478, 0.6248498146341478, 0.6248498146341478, 0.6248498146341478, 0.6248498146341478, 0.6248498146341478, 0.6248498146341478, 0.6248498146341478, 0.5568949684186152, 0.5130424478709298, 0.5130424478709298, 0.473329344248224, 0.4281482349403976, 0.3916113010917321, 0.3916113010917321, 0.36832356080245016, 0.3053362097309802, 0.3053362097309802, 0.3053362097309802, 0.3053362097309802, 0.2648131003033938, 0.21629796268981633, 0.16581648521865394, 0.10343987932552585, 0.10343987932552585, 0.10343987932552585, 0.0056371278021461,

### Ex. 2: Effect of the Initial Global Step‐Size $\sigma_0$
1. Use [2,2] at starting point on the Rastrigin function.
2. Test $\sigma_0\in\{0.1,\,0.5,\,1,\,2,\,5\}$.
3. Record and plot
- final fitness after a fixed budget (e.g. 1000 evals)
- evolution of es.sigma over time (log scale).
4. Explain why too-small and too-large $\sigma_0$ hurt performance, relating findings to the adaptation rule.

In [ ]:
result_dict = {}
sigmas = np.array([0.1, 0.5, 1,2,5])
for sigma in sigmas:
    result = run_cma_es(
        func=rastrigin,
        x0=np.array([2,2]),
        sigma0=sigma,
        max_iterations=250
    )
    result_dict[str(sigma)] = result.best_fitness


### Ex. 3: Visualising the Covariance Matrix Adaptation
1. On Rastrigin (and Sphere) in 2D, log es.C every 5 generations.
2. Use the provided `animate_distribution` to produce a GIF showing the shrinking and rotation of the sampling ellipse.
3. Submit the animation and two short observations about what the animation reveals regarding step-size vs. shape adaptation.

## Part 2: CMA-ES for Reinforcement Learning

So far we have tested CMA-ES on synthetic benchmark functions where we know the true optimum. Now we apply it to a real control task: **LunarLander-v3**.

### The environment
LunarLander-v3 (Box2D) is a classic RL benchmark:
- **Observation**: 8-D continuous vector (lander position, velocity, angle, angular velocity, leg contact flags).
- **Action**: 4 discrete actions (do nothing, fire left engine, fire main engine, fire right engine).
- **Reward**: dense — shaped by distance to landing pad, fuel used, crash/landing bonus.

### Why this problem?
LunarLander with an MLP policy has ~200 parameters, which is:
- large enough that the covariance matrix has real work to do,
- small enough that CMA-ES fits in a few minutes on a laptop,
- non-differentiable end-to-end (argmax over discrete actions), so gradient-based methods cannot be applied naively.

### Policy architecture
A tiny MLP with tanh activations: 8 → 16 → 4 logits, argmax for action selection. Total parameters: `8*16 + 16 + 16*4 + 4 = 212`. We flatten all weights and biases into a single vector `θ ∈ ℝ²¹²` so CMA-ES can treat the whole policy as a point in parameter space.

In [8]:
# Plot 1: Final fitness after 1000 evals for each sigma0
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1a: Final fitness comparison (bar plot)
final_fitness = [result_dict[str(sigma)][-1] for sigma in sigmas]
axes[0].bar(range(len(sigmas)), final_fitness, color='steelblue')
axes[0].set_xlabel(r'$\sigma_0$ value')
axes[0].set_ylabel('Final Best Fitness')
axes[0].set_title('Final Fitness after 1000 Evaluations on Rastrigin')
axes[0].set_xticks(range(len(sigmas)))
axes[0].set_xticklabels([f'{s}' for s in sigmas])
axes[0].grid(True, alpha=0.3)

# Plot 1b: Convergence curves
for sigma in sigmas:
    axes[1].plot(result_dict[str(sigma)], label=f'σ₀={sigma}', linewidth=2)
axes[1].set_xlabel('Generation')
axes[1].set_ylabel('Best Fitness')
axes[1].set_title('CMA-ES Convergence on Rastrigin (Fixed 1000 evals)')
axes[1].set_yscale('log')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Plot 2: Evolution of es.sigma over time (log scale)
plt.figure(figsize=(10, 6))
for sigma in sigmas:
    plt.plot(sigma_history_dict[str(sigma)], label=f'σ₀={sigma}', linewidth=2)
plt.xlabel('Generation')
plt.ylabel('Step-size σ (log scale)')
plt.title('CMA-ES Step-size Adaptation on Rastrigin')
plt.yscale('log')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print summary
print("\n=== Summary: Final fitness for each σ₀ ===")
for sigma, fitness in zip(sigmas, final_fitness):
    print(f"σ₀ = {sigma}: final fitness = {fitness:.3e}")
v3"
OBS_DIM: int = 8
ACT_DIM: int = 4
HIDDEN: int = 16
N_PARAMS: int = OBS_DIM * HIDDEN + HIDDEN + HIDDEN * ACT_DIM + ACT_DIM


class MLPPolicy(nn.Module):
    """Tiny MLP: obs -> hidden (tanh) -> logits -> argmax action."""

    def __init__(self, obs_dim: int = OBS_DIM, hidden: int = HIDDEN, act_dim: int = ACT_DIM) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, act_dim),
        )

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        return self.net(obs)

    @torch.no_grad()
    def act(self, obs: np.ndarray) -> int:
        logits = self.forward(torch.as_tensor(obs, dtype=torch.float32))
        return int(torch.argmax(logits).item())


def set_params(policy: MLPPolicy, flat: np.ndarray) -> None:
    """Copy a flat parameter vector into the policy's weights/biases."""
    assert flat.shape == (N_PARAMS,), f"expected {N_PARAMS} params, got {flat.shape}"
    offset = 0
    for p in policy.parameters():
        n = p.numel()
        p.data.copy_(torch.as_tensor(flat[offset : offset + n], dtype=torch.float32).view_as(p))
        offset += n


def get_params(policy: MLPPolicy) -> np.ndarray:
    """Flatten the policy's weights/biases into a single vector."""
    return np.concatenate([p.detach().cpu().numpy().ravel() for p in policy.parameters()])


def rollout(policy: MLPPolicy, env: gymnasium.Env, seed: int | None = None) -> tuple[float, int]:
    """Run one episode; return (total reward, number of env steps)."""
    obs, _ = env.reset(seed=seed)
    total = 0.0
    steps = 0
    terminated = truncated = False
    while not (terminated or truncated):
        action = policy.act(obs)
        obs, reward, terminated, truncated, _ = env.step(action)
        total += float(reward)
        steps += 1
    return total, steps


def evaluate_policy(
    flat_params: np.ndarray,
    env: gymnasium.Env,
    policy: MLPPolicy,
    n_episodes: int = 3,
    base_seed: int = 0,
) -> tuple[float, int]:
    """Average reward over `n_episodes` with fixed episode seeds (common random numbers).

    Using the same seeds across candidates reduces the noise in pairwise comparisons
    between policies — an antithetic / CRN variance-reduction trick."""
    set_params(policy, flat_params)
    rewards: list[float] = []
    total_steps = 0
    for i in range(n_episodes):
        r, s = rollout(policy, env, seed=base_seed + i)
        rewards.append(r)
        total_steps += s
    return float(np.mean(rewards)), total_steps

### Shared budget

To make the comparison against baselines in Ex. 6 fair, we cap every method at the same number of *environment steps*. This is the currency that matters in RL — wall-clock varies by implementation, but env-steps are identical across methods.

In [9]:
BUDGET_STEPS: int = 500_000
EPISODES_PER_EVAL: int = 3
POPSIZE: int = 16
SIGMA0_RL: float = 0.5

### Ex. 4: Train LunarLander with CMA-ES

1. Create a `MLPPolicy` and a single persistent `gymnasium.make(ENV_NAME)` environment.
2. Write `train_cma_es_rl(...)` that uses `cma.CMAEvolutionStrategy` (ask/tell loop) to **maximize** `evaluate_policy(θ)`. Remember CMA-ES minimizes — negate the reward.
3. Stop when cumulative env-steps exceed `BUDGET_STEPS`.
4. Per generation, record: mean reward of the population, best-so-far reward, cumulative env-steps.
5. Return a dict: `{"steps": [...], "best": [...], "mean": [...], "best_params": np.ndarray}`. We'll reuse this structure for all methods in Ex. 5.
6. Plot reward vs. cumulative env-steps. Render one final rollout with `gymnasium.make(ENV_NAME, render_mode="human")` to see the learned behaviour.

**Hints**
- `popsize=POPSIZE`, `sigma0=SIGMA0_RL`, `x0=np.zeros(N_PARAMS)`.
- `cma.CMAEvolutionStrategy(x0, sigma0, {"popsize": POPSIZE, "verbose": -9})`.
- Use `EPISODES_PER_EVAL` episodes per candidate with the *same* base_seed across the whole population in one generation (CRN). Bump the base_seed each generation so CMA-ES doesn't overfit to three fixed seeds.

In [10]:
# TODO: implement train_cma_es_rl and run it here.
# Expected return shape: dict(steps=list[int], best=list[float], mean=list[float], best_params=np.ndarray)
cma_es_result = None

### Ex. 5: Baselines — is CMA-ES actually doing something clever?

CMA-ES shows nice-looking learning curves, but before we credit the *covariance adaptation* we need a baseline. Two questions:

- **Does search matter at all?** → compare to **random search**: sample i.i.d. Gaussian parameter vectors, keep the best. Same evaluation protocol, same env-step budget.
- **Does being an RL algorithm matter?** → compare to **PPO** from Stable-Baselines3, a standard on-policy actor-critic method. Same env, same env-step budget.

Plot all three on a single chart: *best-so-far reward* (y) vs. *cumulative env-steps* (x). That is the only fair axis — not generations, not wall-clock.

#### Ex. 5a: Random search baseline

Implement `train_random_search(...)` that draws candidates from $\mathcal{N}(0, \sigma_0^2 I)$ and keeps the best, using `evaluate_policy` and `BUDGET_STEPS`. Return the same dict format as `train_cma_es_rl`.

In [11]:
# TODO: implement train_random_search and run it here.
random_result = None

#### Ex. 5b: PPO baseline (Stable-Baselines3)

PPO is a policy-gradient method; it trains by sampling actions stochastically, computing advantages, and taking gradient steps on a clipped surrogate objective. It requires a *differentiable* stochastic policy — so SB3 uses its own MLP underneath, with a softmax over actions and an independent value head.

Install once: `pip install stable-baselines3`.

In [12]:
import tempfile
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.monitor import Monitor


def train_ppo(total_steps: int = BUDGET_STEPS, eval_freq: int = 5_000, seed: int = 0) -> dict:
    """Train PPO on LunarLander-v3. Requires stable-baselines3 >= 2.0 (native Gymnasium support).

    Note: `EvalCallback` only populates `evaluations_timesteps` / `evaluations_results`
    when `log_path` is set — otherwise they remain empty lists. We use a tempdir.
    """
    train_env = Monitor(gymnasium.make(ENV_NAME))
    eval_env = Monitor(gymnasium.make(ENV_NAME))
    log_dir = tempfile.mkdtemp(prefix="ppo_eval_")
    model = PPO(
        "MlpPolicy", 
        train_env,
        policy_kwargs=dict(net_arch=[HIDDEN]),
        learning_rate=3e-4, 
        n_steps=1024, 
        batch_size=64, 
        n_epochs=4,
        gamma=0.99, 
        gae_lambda=0.95, 
        clip_range=0.2,
        verbose=0, 
        seed=seed,
    )
    callback = EvalCallback(
        eval_env, n_eval_episodes=EPISODES_PER_EVAL,
        eval_freq=eval_freq, deterministic=True, verbose=0,
        log_path=log_dir,
    )
    model.learn(total_timesteps=total_steps, callback=callback)
    steps = list(callback.evaluations_timesteps)
    mean_rewards = [float(np.mean(r)) for r in callback.evaluations_results]
    best_so_far: list[float] = []; running = -np.inf
    for r in mean_rewards:
        running = max(running, r); best_so_far.append(running)
    train_env.close(); eval_env.close()
    return {"steps": steps, "best": best_so_far, "mean": mean_rewards, "model": model}

ppo_result = train_ppo()
print(f"PPO collected {len(ppo_result['steps'])} evaluation points")

/Users/joannaheldak/venvs/.notebooks_DS_common/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


PPO collected 100 evaluation points


#### Ex. 5c: Compare all three

Plot best-so-far reward vs. cumulative env-steps for CMA-ES, Random Search, and PPO on the same chart. Add a horizontal line at reward = 200 (the "solved" threshold).

Then answer in 3–5 sentences:
1. Did CMA-ES beat Random Search? If yes — what did the covariance adaptation buy us?
2. How does CMA-ES compare to PPO at the same env-step budget?
3. In what sense is the comparison *unfair* in each direction? (Think: hyperparameter count, bootstrapping in PPO, env-step definition, wall-clock.)

In [13]:
plt.figure(figsize=(10, 6))
plt.plot(cma_es_result["steps"], cma_es_result["best"], label="CMA-ES", linewidth=2)
plt.plot(random_result["steps"], random_result["best"], label="Random Search", linewidth=2)
plt.plot(ppo_result["steps"],   ppo_result["best"],   label="PPO", linewidth=2)
plt.axhline(200, color="gray", linestyle="--", label="Solved threshold (200)")
plt.xlabel("Cumulative environment steps")
plt.ylabel("Best-so-far mean reward")
plt.title("LunarLander-v3: CMA-ES vs. Random Search vs. PPO")
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

TypeError: 'NoneType' object is not subscriptable

<Figure size 1000x600 with 0 Axes>

### When is CMA-ES a good fit?


**When to reach for CMA-ES in practice**: black-box or non-differentiable objective; small-to-medium parameter count (up to a few thousand); reward signal is noisy; you have parallel compute; you do not want to tune many hyperparameters. Classic applications: controller tuning, hyperparameter optimization, evolving neural network weights for control tasks, direct optimization of simulator-based designs.

In [ ]:
# Watch the trained CMA-ES policy (optional — requires a display)
render_env = gymnasium.make(ENV_NAME, render_mode="human")
policy = MLPPolicy(); set_params(policy, cma_es_result["best_params"])
for i in range(3):
    r, _ = rollout(policy, render_env, seed=10_000 + i)
    print(f"Episode {i}: reward={r:.1f}")
render_env.close()